# Recurrent Neural Networks

**Prerequisites**

- MLPs and backpropagation
- PyTorch basics (`nn.Module`, `DataLoader`)

**Outcomes**

- Understand the equations behind the simple (Elman) RNN
- Implement an RNN forward pass from scratch in NumPy
- Understand how vanishing gradients motivate LSTM and GRU
- Implement and train RNN, LSTM, and GRU models in PyTorch
- Benchmark all three on hourly ERCOT energy demand forecasting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)
%matplotlib inline

## Sequence Modeling

- Most data we have studied so far is **iid**: each observation is drawn independently
- Many real problems involve **sequences** where order matters and observations are correlated over time:
    - Natural language: each word depends on the preceding words
    - Audio: each sample depends on the waveform history
    - Energy demand: load at each hour depends on recent consumption, time of day, and day of week
- A standard MLP maps a fixed-size input to an output — it has no natural way to process a variable-length sequence or share parameters across positions in a sequence
- **Recurrent neural networks (RNNs)** address this by maintaining a **hidden state** that is updated at each timestep, carrying information forward through the sequence

## The Simple (Elman) RNN

- At each timestep $t$, the model reads input $x_t \in \mathbb{R}^d$ and updates a **hidden state** $h_t \in \mathbb{R}^n$:

$$h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b_h)$$

- An output is then produced from the hidden state:

$$\hat{y}_t = W_{hy}\, h_t + b_y$$

- Parameters $W_{xh} \in \mathbb{R}^{n \times d}$, $W_{hh} \in \mathbb{R}^{n \times n}$, $W_{hy} \in \mathbb{R}^{k \times n}$ are **shared across all timesteps** — the same weights process every step of the sequence
- $h_t$ acts as a compressed memory of all inputs seen up to time $t$: $h_t = f(x_1, \ldots, x_t)$
- $h_0$ is typically initialized to the zero vector

### Unrolled computation graph

- We can visualize the RNN "unrolled" over $T$ steps:

```
x_1      x_2      x_3           x_T
 |        |        |              |
[h_1] -> [h_2] -> [h_3] -> ... -> [h_T] -> ŷ_T
```

- Training uses **backpropagation through time (BPTT)**: gradients flow backwards through the entire unrolled graph
- BPTT is identical to standard backprop, just applied to a computation graph unrolled over time

### NumPy Implementation

- Before using PyTorch, let's implement the RNN forward pass from scratch in NumPy
- This makes the parameter shapes and data flow concrete

In [ ]:
class RNNNumpy:
    """Simple Elman RNN — forward pass only."""

    def __init__(self, d_in, d_hid, d_out, seed=0):
        rng   = np.random.default_rng(seed)
        scale = 0.1
        # W_xh : input -> hidden   (n x d)
        # W_hh : hidden -> hidden  (n x n)
        # W_hy : hidden -> output  (k x n)
        self.W_xh = rng.standard_normal((d_hid, d_in))  * scale
        self.W_hh = rng.standard_normal((d_hid, d_hid)) * scale
        self.b_h  = np.zeros(d_hid)
        self.W_hy = rng.standard_normal((d_out, d_hid)) * scale
        self.b_y  = np.zeros(d_out)

    def forward(self, xs, h0=None):
        """
        xs : (T, d_in)  — input sequence
        Returns hidden states (T, d_hid) and outputs (T, d_out)
        """
        T = len(xs)
        h = np.zeros(self.b_h.shape) if h0 is None else h0
        hs = np.zeros((T, h.shape[0]))
        ys = np.zeros((T, self.b_y.shape[0]))
        for t, x in enumerate(xs):
            h     = np.tanh(self.W_xh @ x + self.W_hh @ h + self.b_h)
            hs[t] = h
            ys[t] = self.W_hy @ h + self.b_y
        return hs, ys


# ---- Demonstrate on a noisy sine wave ----
T   = 200
t   = np.linspace(0, 4 * np.pi, T)
seq = (np.sin(t) + 0.1 * np.random.randn(T)).reshape(T, 1)

rnn_np = RNNNumpy(d_in=1, d_hid=16, d_out=1)
hs, ys = rnn_np.forward(seq)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t, seq,  label='Input')
axes[0].plot(t, ys,   label='Output (random weights)', alpha=0.8)
axes[0].set_title('RNN forward pass — untrained')
axes[0].set_xlabel('Time')
axes[0].legend()

axes[1].plot(t, hs[:, :4])
axes[1].set_title('First 4 hidden-state dimensions')
axes[1].set_xlabel('Time')

plt.tight_layout()
plt.show()

## Vanishing Gradients

- Training an RNN via BPTT requires computing gradients of the loss with respect to **early** hidden states
- The gradient of $h_t$ with respect to $h_{t-k}$ is a product of $k$ Jacobians:

$$\frac{\partial h_t}{\partial h_{t-k}} = \prod_{i=1}^{k} \frac{\partial h_{t-i+1}}{\partial h_{t-i}} = \prod_{i=1}^{k} \mathrm{diag}\!\left(1 - h_{t-i+1}^2\right) W_{hh}$$

- If the singular values of $W_{hh}$ are less than 1, this product **shrinks exponentially** with $k$ — the **vanishing gradient problem**
- If they are greater than 1, gradients **explode**
- In practice, vanilla RNNs struggle to learn dependencies spanning more than ~10–20 steps
- **Solution**: architectures with explicit, protected memory channels so that gradients can flow backward through time *without* passing through $W_{hh}$ at every step

## Long Short-Term Memory (LSTM)

Hochreiter & Schmidhuber (1997) introduced the LSTM to solve the vanishing gradient problem. The key innovation is a **cell state** $c_t$ that runs alongside the hidden state $h_t$, carrying information across long spans of time with minimal modification.

Writing $z_t = [h_{t-1},\, x_t]$ for the concatenated input, the LSTM computes at each step:

$$f_t = \sigma(W_f z_t + b_f) \qquad \text{(forget gate)}$$

$$i_t = \sigma(W_i z_t + b_i) \qquad \text{(input gate)}$$

$$\tilde{c}_t = \tanh(W_c z_t + b_c) \qquad \text{(candidate cell)}$$

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \qquad \text{(cell state update)}$$

$$o_t = \sigma(W_o z_t + b_o) \qquad \text{(output gate)}$$

$$h_t = o_t \odot \tanh(c_t) \qquad \text{(hidden state)}$$

- **Forget gate $f_t \in (0,1)^n$**: element-wise fraction of $c_{t-1}$ to retain. $f_t \approx 1$ preserves memory; $f_t \approx 0$ erases it
- **Input gate $i_t$**: controls how much of the new candidate $\tilde{c}_t$ to write into the cell
- **Cell state $c_t$**: a "conveyor belt" — when $f_t \approx 1$ and $i_t \approx 0$, $c_t \approx c_{t-1}$ and the gradient of the loss flows backward through the $c_t$ path **without** passing through a saturating nonlinearity
- **Output gate $o_t$**: filters the cell state to produce the hidden state passed on to subsequent layers or timesteps

## Gated Recurrent Unit (GRU)

Cho et al. (2014) proposed the GRU as a simpler alternative to the LSTM. It merges the cell and hidden states into a single $h_t$ and uses only two gates:

$$r_t = \sigma(W_r [h_{t-1},\, x_t] + b_r) \qquad \text{(reset gate)}$$

$$z_t = \sigma(W_z [h_{t-1},\, x_t] + b_z) \qquad \text{(update gate)}$$

$$\tilde{h}_t = \tanh\!\left(W_h [r_t \odot h_{t-1},\; x_t] + b_h\right) \qquad \text{(candidate)}$$

$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

- **Reset gate $r_t$**: how much of the previous hidden state to use when computing the candidate update. $r_t \approx 0$ lets the model start fresh at this step
- **Update gate $z_t$**: interpolates between the old hidden state and the new candidate. $z_t \approx 0$ keeps $h_t \approx h_{t-1}$, acting like LSTM's combined forget + input gates
- GRU has **fewer parameters** than LSTM (no separate cell state, no output gate)
- In practice, GRU and LSTM perform similarly; the GRU is often faster to train and easier to tune

## Application: ERCOT Hourly Energy Demand

- We apply all three architectures to one-step-ahead forecasting of hourly electricity demand in Texas (ERCOT)
- **Task**: given the past $T_\text{seq} = 168$ hours (one week) of load, predict the next hour's load
- **Split**: first four years of data $\to$ train; remainder $\to$ validation
- **Baseline**: seasonal naive — predict that hour $t$ will repeat hour $t - 168$ (same hour last week)

In [ ]:
df = pd.read_parquet('hourly_load_ercot.parquet')
df.index = pd.to_datetime(df.index)

df = df.sort_index()

print(f'Rows: {len(df):,}')
print(f'Range: {df.index.min().date()} to {df.index.max().date()}')
df.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Full series
axes[0].plot(df.index, df['ERCOT'] / 1e3, lw=0.4)
axes[0].set_ylabel('Load (GW)')
axes[0].set_title('ERCOT hourly load — full series')

# Two-week zoom to show diurnal and weekly cycles
zoom = df.iloc[:24 * 14]
axes[1].plot(zoom.index, zoom['ERCOT'] / 1e3)
axes[1].set_ylabel('Load (GW)')
axes[1].set_title('First two weeks — diurnal and weekly patterns')

plt.tight_layout()
plt.show()

In [ ]:
SEQ_LEN = 168   # one week of hourly lags

# --- Train / val split: first 4 years ---
train_end = df.index.min() + pd.DateOffset(years=4)
df_train  = df[df.index <  train_end]
df_val    = df[df.index >= train_end]

print(f'Train: {df_train.index.min().date()} → {df_train.index.max().date()}  ({len(df_train):,} hours)')
print(f'Val:   {df_val.index.min().date()}   → {df_val.index.max().date()}  ({len(df_val):,} hours)')

# --- Z-score normalisation, fit on train only ---
train_mean = df_train['ERCOT'].mean()
train_std  = df_train['ERCOT'].std()

train_series = ((df_train['ERCOT'] - train_mean) / train_std).values.astype(np.float32)
val_series   = ((df_val['ERCOT']   - train_mean) / train_std).values.astype(np.float32)

# --- Visualise the split ---
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(df_train.index, train_series, lw=0.4, label='Train')
ax.plot(df_val.index,   val_series,   lw=0.4, label='Val', alpha=0.8)
ax.axvline(train_end, color='black', lw=1.5, ls='--', label='Split')
ax.set_ylabel('Normalised load')
ax.set_title('Train / val split')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
class EnergyDataset(Dataset):
    """
    Sliding-window dataset.  Item i = (x, y) where
      x : (seq_len, 1) — past seq_len normalised load values
      y : scalar       — load at the next hour
    """
    def __init__(self, series, seq_len):
        self.data    = torch.from_numpy(series)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len].unsqueeze(-1)  # (seq_len, 1)
        y = self.data[idx + self.seq_len]                       # scalar
        return x, y


BATCH_SIZE = 512

train_ds = EnergyDataset(train_series, SEQ_LEN)
val_ds   = EnergyDataset(val_series,   SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Train sequences: {len(train_ds):,}   Val sequences: {len(val_ds):,}')
x_ex, y_ex = train_ds[0]
print(f'x shape: {x_ex.shape}   y shape: {y_ex.shape}')

## Models

- PyTorch provides `nn.RNN`, `nn.LSTM`, and `nn.GRU` with identical interfaces
- We wrap all three in a single class, selecting the recurrent layer by name
- Architecture: one recurrent layer $\to$ linear output head (predict the next scalar value)
- We use the **final hidden state** as the representation of the entire input sequence

In [ ]:
class SequenceModel(nn.Module):
    """Thin wrapper: nn.RNN / nn.LSTM / nn.GRU + a linear output head."""

    _rnn_types = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}

    def __init__(self, rnn_type, hidden_size=64, num_layers=1):
        super().__init__()
        rnn_cls  = self._rnn_types[rnn_type.lower()]
        self.rnn = rnn_cls(
            input_size=1, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x    : (batch, seq_len, 1)
        # out  : (batch, seq_len, hidden_size)
        # _    : final hidden state (ignored here)
        out, _ = self.rnn(x)
        return self.head(out[:, -1]).squeeze(-1)   # scalar per sequence


# Parameter counts
for name in ['rnn', 'lstm', 'gru']:
    m = SequenceModel(name, hidden_size=64)
    n = sum(p.numel() for p in m.parameters())
    print(f'{name.upper():4s}  parameters: {n:,}')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')


def run_epoch(model, loader, optimizer=None):
    """One forward pass over loader.  Pass optimizer=None for eval mode."""
    is_train = optimizer is not None
    model.train(is_train)
    loss_fn = nn.MSELoss()
    total   = 0.0
    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss = loss_fn(pred, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total += loss.item()
    return total / len(loader)


def fit(model, n_epochs=20, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_hist, val_hist = [], []
    for epoch in range(1, n_epochs + 1):
        tl = run_epoch(model, train_loader, optimizer)
        vl = run_epoch(model, val_loader)
        train_hist.append(tl)
        val_hist.append(vl)
        if epoch % 5 == 0:
            print(f'  epoch {epoch:2d}/{n_epochs}  train={tl:.4f}  val={vl:.4f}')
    return train_hist, val_hist


def val_mae(model):
    """MAE on the validation set, converted back to original MWh units."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in val_loader:
            preds.append(model(x.to(device)).cpu())
            targets.append(y)
    preds   = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    return float(train_mean + np.mean(np.abs(preds - targets)) * train_std)

In [ ]:
N_EPOCHS    = 20
HIDDEN_SIZE = 64

results = {}

for name in ['rnn', 'lstm', 'gru']:
    print(f'\nTraining {name.upper()}...')
    model = SequenceModel(name, hidden_size=HIDDEN_SIZE)
    train_hist, val_hist = fit(model, n_epochs=N_EPOCHS)
    mae = val_mae(model)
    results[name] = {'model': model, 'train': train_hist, 'val': val_hist, 'mae': mae}
    print(f'  Val MAE: {mae:,.0f} MWh')

In [ ]:
# Seasonal naive: predict y[t] = y[t - 168]  (same hour last week)
# In our sliding-window dataset, item i has target = val_series[i + SEQ_LEN]
# and its first input element is val_series[i]  (exactly 168 steps earlier)
naive_preds   = val_series[:-SEQ_LEN]   # y[t - 168]
naive_targets = val_series[SEQ_LEN:]    # y[t]
naive_mae     = float(np.mean(np.abs(naive_preds - naive_targets)) * train_std)

print(f'Seasonal naive MAE : {naive_mae:,.0f} MWh')
for name, res in results.items():
    print(f'{name.upper():4s} val MAE      : {res["mae"]:,.0f} MWh')

In [ ]:
colors = {'rnn': 'steelblue', 'lstm': 'darkorange', 'gru': 'forestgreen'}
epochs = range(1, N_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Validation loss curves ---
for name, res in results.items():
    axes[0].plot(epochs, res['val'], color=colors[name], label=name.upper())
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Val MSE (normalised)')
axes[0].set_title('Validation loss during training')
axes[0].legend()

# --- MAE bar chart ---
labels     = ['Naive'] + [n.upper() for n in results]
maes       = [naive_mae] + [results[n]['mae'] for n in results]
bar_colors = ['lightgray'] + [colors[n] for n in results]
bars = axes[1].bar(labels, maes, color=bar_colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, maes):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 20,
        f'{val:,.0f}',
        ha='center', fontsize=11
    )
axes[1].set_ylabel('Val MAE (MWh)')
axes[1].set_title('Model comparison — validation MAE')

plt.tight_layout()
plt.show()

In [ ]:
# Plot actual vs forecast for the best gated model over two validation weeks
best_name  = min(('rnn', 'lstm', 'gru'), key=lambda n: results[n]['mae'])
best_model = results[best_name]['model'].eval()

n_show  = 24 * 14
x_arr   = np.stack([val_series[i : i + SEQ_LEN] for i in range(n_show)])
x_t     = torch.from_numpy(x_arr).unsqueeze(-1).to(device)   # (n_show, SEQ_LEN, 1)

with torch.no_grad():
    pred_norm = best_model(x_t).cpu().numpy()

target_norm = val_series[SEQ_LEN : SEQ_LEN + n_show]
time_show   = df_val.index[SEQ_LEN : SEQ_LEN + n_show]

# Convert back to MWh
pred_mw   = pred_norm   * train_std + train_mean
target_mw = target_norm * train_std + train_mean

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(time_show, target_mw / 1e3, label='Actual',                 lw=1.5)
ax.plot(time_show, pred_mw   / 1e3, label=f'{best_name.upper()} forecast', lw=1.5, alpha=0.8)
ax.set_ylabel('Load (GW)')
ax.set_title(f'{best_name.upper()}: actual vs one-step-ahead forecast — first 2 weeks of validation')
ax.legend()
plt.tight_layout()
plt.show()

## References

- Elman, J. L. (1990). Finding structure in time. *Cognitive Science*, 14(2), 179–211.
- Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735–1780.
- Cho, K., van Merrienboer, B., Gulcehre, C., Bahdanau, D., Bougares, F., Schwenk, H., & Bengio, Y. (2014). Learning phrase representations using RNN encoder-decoder for statistical machine translation. *EMNLP 2014*.